# データの可視化

In [0]:
!pip install japanize-matplotlib

In [0]:
%restart_python

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import japanize_matplotlib
from pyspark.sql.functions import *


# matplotlib関連の全警告を抑制（フォント警告含む）
warnings.filterwarnings('ignore')

# 日本語表示設定（フォールバックで表示される）
plt.rcParams['axes.unicode_minus'] = False

## データ読み込み

`account`（口座属性ディメンション）と `transaction`（取引明細ファクト、1000万行）を読み込む。`region` は `account` にのみ存在するため、クロス集計には join が必要。ファクト表は集計後の小さい結果のみを `toPandas()` する（生データを丸ごと driver に持ってこない）。

In [ ]:
# 表示順（地域・チャネル・時間帯を意味のある順序で固定するため）
REGION_ORDER = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州・沖縄"]
CHANNEL_ORDER = ["ATM", "窓口", "ネットバンキング", "モバイルアプリ"]
TIME_BAND_ORDER = ["深夜(0-5時)", "午前(6-11時)", "午後(12-17時)", "夜(18-23時)"]

df_account_region = spark.read.table("workspace.datasets.account").select("account_id", "region")
df_transaction = spark.read.table("workspace.datasets.transaction")

## クロス集計①: 地域 × チャネル

取引を `account_id` で口座の `region` と結合し、地域×チャネル別の取引件数を Spark 側で集計する。

In [ ]:
region_channel_sdf = (
    df_transaction.join(df_account_region, on="account_id", how="inner")
    .groupBy("region", "channel")
    .count()
)

display(region_channel_sdf)

In [ ]:
pivot_channel = (
    region_channel_sdf.toPandas()
    .pivot(index="region", columns="channel", values="count")
    .reindex(index=REGION_ORDER, columns=CHANNEL_ORDER)
    .fillna(0)
    .astype(int)
)

pivot_channel

## クロス集計②: 地域 × 時間帯

`transaction_timestamp` から `hour()` で時間帯（6時間区切りの4バンド）を導出し、地域×時間帯別の取引件数を集計する。

※ 本データセットの `transaction_timestamp` は過去1年間の一様乱数で生成されているため、実際の日内パターン（営業時間の偏りなど）は再現されない。ここではあくまでクロス集計・ヒートマップ可視化の技術デモとして扱う。

In [ ]:
region_timeband_sdf = (
    df_transaction.join(df_account_region, on="account_id", how="inner")
    .withColumn(
        "time_band",
        when(hour("transaction_timestamp").between(0, 5), lit("深夜(0-5時)"))
        .when(hour("transaction_timestamp").between(6, 11), lit("午前(6-11時)"))
        .when(hour("transaction_timestamp").between(12, 17), lit("午後(12-17時)"))
        .otherwise(lit("夜(18-23時)"))
    )
    .groupBy("region", "time_band")
    .count()
)

display(region_timeband_sdf)

In [ ]:
pivot_timeband = (
    region_timeband_sdf.toPandas()
    .pivot(index="region", columns="time_band", values="count")
    .reindex(index=REGION_ORDER, columns=TIME_BAND_ORDER)
    .fillna(0)
    .astype(int)
)

pivot_timeband

## 可視化ダッシュボード

地域を行軸に持つ2つのクロス集計を、1つのダッシュボード図として並べて表示する。

In [ ]:
def plot_crosstab_heatmap(ax, pivot_df, title, xlabel, cmap="Blues"):
    data = pivot_df.values
    im = ax.imshow(data, cmap=cmap, aspect="auto")

    ax.set_xticks(range(len(pivot_df.columns)))
    ax.set_xticklabels(pivot_df.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(pivot_df.index)))
    ax.set_yticklabels(pivot_df.index)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("地域")

    vmax = data.max()
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            ax.text(
                j, i, f"{value:,}",
                ha="center", va="center",
                color="white" if value > vmax * 0.6 else "black",
                fontsize=9,
            )

    ax.figure.colorbar(im, ax=ax, label="取引件数")
    return im

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("地域別 取引状況ダッシュボード", fontsize=18, fontweight="bold")

plot_crosstab_heatmap(axes[0], pivot_channel, "地域 × チャネル別 取引件数", "チャネル")
plot_crosstab_heatmap(axes[1], pivot_timeband, "地域 × 時間帯別 取引件数", "時間帯")

plt.tight_layout()
plt.show()